# Scoring and Rescue Workflow

This notebook picks up after clustering and rebuilds the note-level scoring logic in a reusable way. It separates three concepts that were previously tangled together:
- note-level score aggregation
- tweet-level model simulation
- analysis tables used later by topics and figures

## Outputs

Running this notebook will create:
- `data/processed/scores.parquet`
- `data/processed/final_table.parquet`
- `data/processed/rescue_summary.parquet`
- `data/processed/pluralistic_breakdown.parquet`
- `data/processed/selection_log.parquet`
- `data/processed/selection_status_summary.parquet`
- `data/processed/diagnostic_notes.parquet` *(top-N notes ranked by between-cluster approval spread; consumed by the topic-modeling stage to restrict topic estimation to cluster-divergent notes)*

These are the main reusable artifacts for the rest of the project.

In [ ]:
from pathlib import Path
import sys
import numpy as np

def find_project_root() -> Path:
    """Anchor on config.txt — safe from any subdir depth."""
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "config.txt").exists():
            return p
    raise RuntimeError("config.txt not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from src.config import ScoringConfig
from src.io import get_config_int, get_interim_dir, get_processed_dir, get_test_mode, load_ratings_with_final_cluster, load_table, save_table
from src.scoring import (
    build_analysis_table,
    build_model_counts,
    build_pluralistic_breakdown,
    build_selection_status_summary,
    compute_note_scores,
    select_diagnostic_notes,
    simulate_models,
    summarize_model_selections,
)
from src.plots import plot_consensus_map, set_notebook_plot_style

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 1000)
set_notebook_plot_style()

TEST_MODE = get_test_mode()
print(f'=== TEST_MODE = {TEST_MODE}'
      f' (' + ('smoke test (small sample)' if TEST_MODE else 'FULL DATA') + ') ===')


## Configuration

In [ ]:
config = ScoringConfig(
    bridge_threshold=0.5,
    min_note_ratings=3,
    eps=1e-6,
)

INTERIM_DIR = get_interim_dir()
PROCESSED_DIR = get_processed_dir()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## Load Inputs

The scoring notebook now runs entirely from the clustering output. That means it no longer reloads `master_sample.csv`. The single required upstream table is `ratings_clustered.parquet`, which already contains the filtered rating rows, note metadata, and user cluster assignments.

In [ ]:
ratings_clustered = load_ratings_with_final_cluster()

print(f'Clustered ratings slice: {len(ratings_clustered):,}')
print(f'Unique notes: {ratings_clustered["noteId"].nunique():,}')
print(f'Unique tweets: {ratings_clustered["tweetId"].nunique():,}')
print(f'Unique clustered users: {ratings_clustered["raterParticipantId"].nunique():,}')

## Build Note-Level Scores

This table is the analytical core of the scoring stage. Each row is one note, and the columns answer four different questions:
- `global_approval`: how the note performs when all clustered ratings are pooled together
- `cluster_*_approval`: how the same note performs inside each latent user cluster
- `bridge_score`: the geometric-mean consensus score used for the representative selector (only computed for notes with at least `min_cluster_ratings` raters per cluster; otherwise `NaN`)
- disagreement features such as `disagreement_std` and `approval_gap_0_1`: how much approval diverges across clusters

When a cluster has zero observed raters for a note, cluster-specific approval falls back to 0.5 (midpoint neutrality). However, bridge score requires at least 3 raters **per cluster** to prevent the fallback from inflating cross-cluster agreement.

In [ ]:
scores = compute_note_scores(
    df_clustered=ratings_clustered,
    config=config,
)

display(scores.head())
print('Scored notes:', len(scores))

## Build the Reusable Analysis Table

`final_table` is a presentation-friendly version of `scores`. It keeps the same analytical content, but renames the main fields so they are easier to inspect, filter, join to topic outputs, and later send into the LLM context-classification stage.

Use it when you want to browse results manually or connect scoring outputs to downstream notebooks without rebuilding transformations.

In [ ]:
final_table = build_analysis_table(scores)
display(final_table.head())

## Simulate Simple Majoritarian Rule, Pluralistic, and Representative Models

This section moves from note-level scoring to tweet-level decision rules, but the central question is now rescue-oriented: which suppressed notes in the NMR pool can plausibly be surfaced?

Each model asks a different question:
- `Simple Majoritarian Rule`: which single note would a simple pooled-approval majority rule surface when we ask what a homogeneous crowd would elevate from the NMR pool, abstracting away from X's much more complex live algorithm?
- `Pluralistic-K`: which note would each cluster prefer if every cluster could surface its own pick?
- `Representative`: which single note best rescues NMR notes by balancing approval across clusters through the bridge score?

The outputs below tell you both coverage and composition: not only how many tweets each strategy reaches, but also whether the selected notes were already marked helpful or were still stuck in NMR.


In [ ]:
rescue_summary = simulate_models(scores, bridge_threshold=config.bridge_threshold)
model_counts = build_model_counts(rescue_summary)
pluralistic_breakdown = build_pluralistic_breakdown(scores)
selection_log = summarize_model_selections(scores, bridge_threshold=config.bridge_threshold)
selection_status_summary = build_selection_status_summary(selection_log)

print(model_counts)
display(rescue_summary.head())
display(pluralistic_breakdown)
display(selection_status_summary.sort_values(['strategy', 'status_group']))

## Strategy Diagnostics and Selection Composition

The charts below are meant to be more decision-oriented than the original exploratory plots, with a particular focus on NMR rescue rather than generic model comparison.

Read them as follows:
- the first chart shows how many picks each strategy makes and how those picks split between notes already labeled `Helpful` and notes still labeled `NMR`
- the second chart compares the average `global_approval` and average `bridge_score` of the notes selected by each strategy
- the third chart shows how often each strategy converges with the Simple Majoritarian Rule and chooses the same note for a tweet
- the scatter plot maps all notes in the `cluster_0_approval` x `cluster_1_approval` space and highlights which notes are selected by pluralistic and representative rules


In [ ]:
theme = {
    'simple_majoritarian': '#4b5563',
    'pluralistic': '#d97706',
    'representative': '#059669',
    'cluster0': '#2563eb',
    'cluster1': '#7c3aed',
    'helpful': '#0f766e',
    'nmr': '#dc2626',
    'other': '#9ca3af',
    'grid': '#d1d5db',
    'all_notes': '#cbd5e1',
}

strategy_order = ['Simple Majoritarian Rule', 'Pluralistic-K', 'Representative']
strategy_order += sorted([s for s in selection_status_summary['strategy'].unique() if s.startswith('Cluster ')])
status_order = ['Helpful', 'NMR', 'Other']
status_palette = {
    'Helpful': theme['helpful'],
    'NMR': theme['nmr'],
    'Other': theme['other'],
}

plot_summary = selection_status_summary.copy()
plot_summary['strategy'] = pd.Categorical(plot_summary['strategy'], categories=strategy_order, ordered=True)
plot_summary['status_group'] = pd.Categorical(plot_summary['status_group'], categories=status_order, ordered=True)
plot_summary = plot_summary.sort_values(['strategy', 'status_group'])

stacked = (
    plot_summary.pivot_table(
        index='strategy',
        columns='status_group',
        values='selected_picks',
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(strategy_order)
    .fillna(0)
)

approval_summary = (
    selection_log.groupby('strategy')
    .agg(
        avg_global_approval=('global_approval', 'mean'),
        avg_bridge_score=('bridge_score', 'mean'),
        selected_picks=('selected_noteId', 'count'),
        unique_notes=('selected_noteId', 'nunique'),
    )
    .reset_index()
)
approval_summary['strategy'] = pd.Categorical(approval_summary['strategy'], categories=strategy_order, ordered=True)
approval_summary = approval_summary.sort_values('strategy')

simple_majoritarian_map = (
    selection_log[selection_log['strategy'] == 'Simple Majoritarian Rule'][['tweetId', 'selected_noteId']]
    .drop_duplicates('tweetId')
    .rename(columns={'selected_noteId': 'simple_majoritarian_noteId'})
)

overlap_rows = []
for strategy in [s for s in strategy_order if s != 'Simple Majoritarian Rule']:
    strategy_df = selection_log[selection_log['strategy'] == strategy].copy()
    merged = strategy_df.merge(simple_majoritarian_map, on='tweetId', how='inner')
    if merged.empty:
        continue
    overlap_rows.append({
        'strategy': strategy,
        'overlap_rate': (merged['selected_noteId'] == merged['simple_majoritarian_noteId']).mean(),
        'matching_tweets': int((merged['selected_noteId'] == merged['simple_majoritarian_noteId']).sum()),
        'total_rows': len(merged),
    })

overlap_summary = pd.DataFrame(overlap_rows)
if not overlap_summary.empty:
    overlap_summary['strategy'] = pd.Categorical(overlap_summary['strategy'], categories=[s for s in strategy_order if s != 'Simple Majoritarian Rule'], ordered=True)
    overlap_summary = overlap_summary.sort_values('strategy')

fig = plt.figure(figsize=(22, 18))
gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1], hspace=0.32, wspace=0.18)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

bottom = None
for status in status_order:
    values = stacked[status].values if status in stacked.columns else [0] * len(stacked.index)
    ax1.bar(
        stacked.index.astype(str),
        values,
        bottom=bottom,
        color=status_palette[status],
        edgecolor='white',
        linewidth=1.5,
        label=status,
        width=0.72,
    )
    bottom = values if bottom is None else bottom + values

for idx, strategy in enumerate(stacked.index.astype(str)):
    total = int(stacked.loc[strategy].sum())
    helpful = int(stacked.loc[strategy, 'Helpful']) if 'Helpful' in stacked.columns else 0
    nmr = int(stacked.loc[strategy, 'NMR']) if 'NMR' in stacked.columns else 0
    ax1.text(idx, total + max(1, total * 0.015), f'{total} picks\nH={helpful} | NMR={nmr}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.set_title('Selection Composition by Strategy', fontsize=20, fontweight='bold', pad=18)
ax1.set_ylabel('Selected picks', fontsize=14)
ax1.grid(axis='y', color=theme['grid'], alpha=0.6)
ax1.legend(title='Current system label', frameon=False, ncol=3, loc='upper right')
ax1.tick_params(axis='x', rotation=15, labelsize=12)

long_approval = approval_summary.melt(
    id_vars=['strategy', 'selected_picks', 'unique_notes'],
    value_vars=['avg_global_approval', 'avg_bridge_score'],
    var_name='metric',
    value_name='value',
)
metric_labels = {
    'avg_global_approval': 'Avg global approval',
    'avg_bridge_score': 'Avg bridge score',
}
long_approval['metric'] = long_approval['metric'].map(metric_labels)

sns.barplot(
    data=long_approval,
    x='strategy',
    y='value',
    hue='metric',
    palette=['#1d4ed8', '#059669'],
    ax=ax2,
)
ax2.set_title('Average Approval Profile of Selected Notes', fontsize=17, fontweight='bold', pad=14)
ax2.set_xlabel('')
ax2.set_ylabel('Average score', fontsize=13)
ax2.set_ylim(0, 1)
ax2.grid(axis='y', color=theme['grid'], alpha=0.6)
ax2.tick_params(axis='x', rotation=20, labelsize=11)
ax2.legend(title='', frameon=False, loc='upper right')

if not overlap_summary.empty:
    sns.barplot(
        data=overlap_summary,
        x='strategy',
        y='overlap_rate',
        palette=['#d97706', '#059669', '#2563eb', '#7c3aed'][:len(overlap_summary)],
        ax=ax3,
    )
    for idx, row in overlap_summary.reset_index(drop=True).iterrows():
        ax3.text(
            idx,
            row['overlap_rate'] + 0.015,
            f"{row['overlap_rate']:.1%}\n({int(row['matching_tweets'])}/{int(row['total_rows'])})",
            ha='center',
            va='bottom',
            fontsize=11,
            fontweight='bold',
        )
    ax3.set_ylim(0, 1.05)
    ax3.set_title('Selection Overlap with Simple Majoritarian Rule', fontsize=17, fontweight='bold', pad=14)
    ax3.set_xlabel('')
    ax3.set_ylabel('Share of selections matching the Simple Majoritarian Rule note', fontsize=13)
    ax3.grid(axis='y', color=theme['grid'], alpha=0.6)
    ax3.tick_params(axis='x', rotation=20, labelsize=11)
else:
    ax3.text(0.5, 0.5, 'No overlap summary available', ha='center', va='center', fontsize=14)
    ax3.set_axis_off()

plt.show()

cluster_x = 'cluster_0_approval'
cluster_y = 'cluster_1_approval'
if cluster_x in scores.columns and cluster_y in scores.columns:
    simple_majoritarian_notes = set(selection_log.loc[selection_log['strategy'] == 'Simple Majoritarian Rule', 'selected_noteId'])
    representative_notes = set(selection_log.loc[selection_log['strategy'] == 'Representative', 'selected_noteId'])
    pluralistic_notes = set(selection_log.loc[selection_log['strategy'] == 'Pluralistic-K', 'selected_noteId'])

    scatter_df = scores[[
        'noteId', 'total_votes', 'global_approval', 'bridge_score', cluster_x, cluster_y
    ]].drop_duplicates('noteId').copy()
    scatter_df['selection_group'] = 'Other notes'
    scatter_df.loc[scatter_df['noteId'].isin(simple_majoritarian_notes), 'selection_group'] = 'Simple Majoritarian Rule picks'
    scatter_df.loc[scatter_df['noteId'].isin(pluralistic_notes), 'selection_group'] = 'Pluralistic picks'
    scatter_df.loc[scatter_df['noteId'].isin(representative_notes), 'selection_group'] = 'Representative picks'

    scatter_palette = {
        'Other notes': theme['all_notes'],
        'Simple Majoritarian Rule picks': theme['simple_majoritarian'],
        'Pluralistic picks': theme['pluralistic'],
        'Representative picks': theme['representative'],
    }
    scatter_sizes = scatter_df['total_votes'].clip(lower=3, upper=50) * 2

    fig, ax = plt.subplots(figsize=(14, 11))
    sns.scatterplot(
        data=scatter_df,
        x=cluster_x,
        y=cluster_y,
        hue='selection_group',
        hue_order=['Other notes', 'Simple Majoritarian Rule picks', 'Pluralistic picks', 'Representative picks'],
        palette=scatter_palette,
        size=scatter_sizes,
        sizes=(20, 220),
        alpha=0.78,
        linewidth=0.4,
        edgecolor='white',
        ax=ax,
    )
    ax.plot([0, 1], [0, 1], linestyle='--', color='#64748b', linewidth=1.5, alpha=0.7)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Cluster 0 approval', fontsize=14)
    ax.set_ylabel('Cluster 1 approval', fontsize=14)
    ax.set_title('Selected Notes in Cluster Approval Space', fontsize=19, fontweight='bold', pad=16)
    ax.grid(color=theme['grid'], alpha=0.55)
    ax.legend(title='Selection group', frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0))
    plt.tight_layout()
    plt.show()

fig, ax = plot_consensus_map(scores, sample_size=3000)
fig.set_size_inches(14, 10)
ax.set_title('Consensus Map: Mean Approval vs Disagreement', fontsize=18, fontweight='bold', pad=16)
ax.grid(color=theme['grid'], alpha=0.6)
plt.show()

## Cluster-Diagnostic Note Subset

To keep topic modeling tractable on the full-data slice, we restrict the topic input to the notes that most strongly differentiate user clusters. For each note we compute the spread between the highest and lowest cluster-conditional approval rate, then keep the top-N notes by this spread.

The threshold `FULL_DIAGNOSTIC_TOP_N` (default 10,000) is set ex ante in `config.txt` so the subset is fixed before any topic results are inspected. Notes that fail the per-cluster minimum rater count are excluded so the spread reflects real cluster signal rather than the 0.5 neutral-approval fallback.

This subset is the input the topic-modeling notebook expects; if `diagnostic_notes.parquet` is missing, the topics stage falls back to the full `scores` frame.

In [ ]:
diagnostic_min_spread = float(get_config_int('FULL_DIAGNOSTIC_MIN_SPREAD_PCT', 90)) / 100
diagnostic_min_cluster_ratings = get_config_int('FULL_DIAGNOSTIC_MIN_CLUSTER_RATINGS', config.min_note_ratings)

diagnostic_notes = select_diagnostic_notes(
    scores,
    min_spread=diagnostic_min_spread,
    min_cluster_ratings=diagnostic_min_cluster_ratings,
    require_summary=True,
)

print(f'Diagnostic spread threshold: {diagnostic_min_spread:.2f}')
print(f'Min per-cluster rater count: {diagnostic_min_cluster_ratings}')
print(f'Notes passing threshold: {len(diagnostic_notes):,}')
if len(diagnostic_notes) > 0:
    print(
        f'Spread range: [{diagnostic_notes["cluster_approval_spread"].min():.3f}, '
        f'{diagnostic_notes["cluster_approval_spread"].max():.3f}]'
    )
    print(f'Spread median: {diagnostic_notes["cluster_approval_spread"].median():.3f}')


## Persist Scoring Artifacts

In [ ]:
save_table(scores, PROCESSED_DIR / 'scores.parquet')
save_table(final_table, PROCESSED_DIR / 'final_table.parquet')
save_table(rescue_summary, PROCESSED_DIR / 'rescue_summary.parquet')
save_table(pluralistic_breakdown, PROCESSED_DIR / 'pluralistic_breakdown.parquet')
save_table(selection_log, PROCESSED_DIR / 'selection_log.parquet')
save_table(selection_status_summary, PROCESSED_DIR / 'selection_status_summary.parquet')
save_table(diagnostic_notes, PROCESSED_DIR / 'diagnostic_notes.parquet')

## Why This Split Matters for the Next Step

Your next phase is LLM-based context classification. This notebook now gives you a stable handoff dataset with text, metadata, cluster scores, bridge score, and disagreement features already aligned at the note level.

## B. Bridge Score Robustness

Two targeted robustness checks for the bridge score design.

- **B1** tests whether the rescue counts change if we replace geometric mean with harmonic mean or the minimum operator.
- **B2** tests whether the ε = 10⁻⁶ floor is arbitrary or inconsequential.

### B1 — Alternative Aggregation Functions

The geometric mean is one defensible choice for combining cluster approvals, but not the only one. We re-run the representative rescue simulation under two alternatives:

| Function | Formula | Behaviour |
|---|---|---|
| Geometric mean *(baseline)* | exp(mean(log(aᵢ))) | Penalises low values; smooth |
| Harmonic mean | n / Σ(1/aᵢ) | Penalises low values more aggressively than geometric |
| Minimum operator | min(a₀, a₁) | Hard veto — worst-cluster approval decides |

In [ ]:
from src.scoring import get_cluster_approval_columns

cluster_cols_b = get_cluster_approval_columns(scores)
EPS_B = config.eps

# --- Alternative bridge score functions ---
def harmonic_mean_bridge(df, eps=EPS_B):
    clipped = df[cluster_cols_b].clip(eps, 1.0)
    n = len(cluster_cols_b)
    return n / (1.0 / clipped).sum(axis=1)

def min_bridge(df):
    return df[cluster_cols_b].min(axis=1)

scores_b1 = scores.copy()
scores_b1['bridge_geom']     = scores_b1['bridge_score']          # baseline
scores_b1['bridge_harmonic'] = harmonic_mean_bridge(scores_b1)
scores_b1['bridge_min']      = min_bridge(scores_b1)

# --- Per-tweet top-note selector ---
def get_top_note_per_tweet(df, bridge_col):
    return (
        df.sort_values(bridge_col, ascending=False)
        .groupby('tweetId')
        .first()[['noteId', 'currentStatus', bridge_col]]
    )

# --- Count representative rescues for a given bridge column ---
def count_rep_rescues(df, bridge_col, threshold=config.bridge_threshold):
    top = get_top_note_per_tweet(df, bridge_col)
    rescued = (
        (top[bridge_col] > threshold) &
        (top['currentStatus'] != 'CURRENTLY_RATED_HELPFUL')
    ).sum()
    return int(rescued)

rescued_geom     = count_rep_rescues(scores_b1, 'bridge_geom')
rescued_harmonic = count_rep_rescues(scores_b1, 'bridge_harmonic')
rescued_min      = count_rep_rescues(scores_b1, 'bridge_min')

# --- Selection overlap with baseline ---
sel_geom     = get_top_note_per_tweet(scores_b1, 'bridge_geom')['noteId']
sel_harmonic = get_top_note_per_tweet(scores_b1, 'bridge_harmonic')['noteId']
sel_min      = get_top_note_per_tweet(scores_b1, 'bridge_min')['noteId']

common = sel_geom.index.intersection(sel_harmonic.index).intersection(sel_min.index)
overlap_harmonic = (sel_geom[common] == sel_harmonic[common]).mean()
overlap_min      = (sel_geom[common] == sel_min[common]).mean()

total_tweets_b1 = scores_b1['tweetId'].nunique()

robustness_b1 = pd.DataFrame({
    'Function':              ['Geometric mean (baseline)', 'Harmonic mean',  'Minimum operator'],
    'Formula':               ['exp(mean(log(aᵢ)))',        'n / Σ(1/aᵢ)',   'min(a₀, a₁)'],
    'Rep. rescues':          [rescued_geom, rescued_harmonic, rescued_min],
    'Rescues / total tweets': [
        f'{rescued_geom     / total_tweets_b1:.1%}',
        f'{rescued_harmonic / total_tweets_b1:.1%}',
        f'{rescued_min      / total_tweets_b1:.1%}',
    ],
    'Note overlap with baseline': ['—', f'{overlap_harmonic:.1%}', f'{overlap_min:.1%}'],
})

print(f'Total tweets: {total_tweets_b1:,}\n')
display(robustness_b1)

# --- Visualisation ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: rescue count bar chart
func_labels = ['Geometric\n(baseline)', 'Harmonic', 'Minimum']
rescues_b1  = [rescued_geom, rescued_harmonic, rescued_min]
colors_b1   = ['#059669', '#d97706', '#7c3aed']
bars = axes[0].bar(func_labels, rescues_b1, color=colors_b1, alpha=0.85,
                   edgecolor='white', linewidth=1.5, width=0.55)
for bar, val in zip(bars, rescues_b1):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 2, str(val),
                 ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[0].set_title('Representative Rescues by Bridge Function', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Rescued tweets (NMR → visible)')
axes[0].set_ylim(0, max(rescues_b1) * 1.18)
axes[0].grid(axis='y', color='#d1d5db', alpha=0.6)

# Right: scatter geometric vs harmonic / minimum (note-level)
sample_b1 = scores_b1.sample(min(2000, len(scores_b1)), random_state=42)
axes[1].scatter(sample_b1['bridge_geom'], sample_b1['bridge_harmonic'],
                alpha=0.35, s=15, color='#d97706', label='Harmonic')
axes[1].scatter(sample_b1['bridge_geom'], sample_b1['bridge_min'],
                alpha=0.35, s=15, color='#7c3aed', label='Minimum', marker='x')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
axes[1].set_title('Note-Level Bridge Score: Geometric vs Alternatives\n(sample n=2,000)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Geometric mean bridge score')
axes[1].set_ylabel('Alternative bridge score')
axes[1].legend(frameon=False)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].grid(color='#d1d5db', alpha=0.5)

plt.tight_layout()
plt.show()

### B2 — Epsilon Sensitivity

The geometric mean clips approvals at ε before taking the log, so notes with zero approval in one cluster are not sent to −∞. The baseline uses ε = 10⁻⁶. Here we test whether varying ε changes the rescue count or the note selections.

In [ ]:
epsilons = [1e-6, 1e-3, 1e-2, 1e-1]

eps_rows    = []
base_sel_b2 = None

for eps_val in epsilons:
    sc_eps = scores.copy()
    sc_eps['bridge_eps'] = np.exp(
        np.log(sc_eps[cluster_cols_b].clip(eps_val, 1.0)).mean(axis=1)
    )

    rescued  = count_rep_rescues(sc_eps, 'bridge_eps')
    top_eps  = get_top_note_per_tweet(sc_eps, 'bridge_eps')['noteId']

    if base_sel_b2 is None:
        base_sel_b2      = top_eps
        overlap_vs_base  = 1.0
    else:
        common_eps      = base_sel_b2.index.intersection(top_eps.index)
        overlap_vs_base = (base_sel_b2[common_eps] == top_eps[common_eps]).mean()

    baseline_rescued = eps_rows[0]['Rep. rescues'] if eps_rows else rescued
    eps_rows.append({
        'ε':                          eps_val,
        'ε (display)':               f'{eps_val:.0e}',
        'Rep. rescues':               rescued,
        'Δ vs ε=1e-6':               rescued - baseline_rescued,
        'Note overlap vs ε=1e-6':    f'{overlap_vs_base:.3%}',
    })

eps_df = pd.DataFrame(eps_rows)
print('B2: Epsilon sensitivity for geometric mean bridge score\n')
display(eps_df[['ε (display)', 'Rep. rescues', 'Δ vs ε=1e-6', 'Note overlap vs ε=1e-6']])

# How many notes shift bridge score >0.01 between extremes?
sc_low  = scores.copy()
sc_high = scores.copy()
sc_low['bridge_eps']  = np.exp(np.log(scores[cluster_cols_b].clip(1e-6, 1.0)).mean(axis=1))
sc_high['bridge_eps'] = np.exp(np.log(scores[cluster_cols_b].clip(1e-1, 1.0)).mean(axis=1))
n_affected = (abs(sc_high['bridge_eps'] - sc_low['bridge_eps']) > 0.01).sum()
print(f'\nNotes where bridge score shifts >0.01 between ε=1e-6 and ε=0.1: '
      f'{n_affected:,} / {len(scores):,} ({n_affected/len(scores):.1%})')

# --- Visualisation ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: rescue count vs epsilon
eps_labels   = [r['ε (display)'] for r in eps_rows]
rescue_vals  = [r['Rep. rescues'] for r in eps_rows]
axes[0].plot(eps_labels, rescue_vals, marker='o', color='#059669',
             linewidth=2.2, markersize=9)
for i, (lbl, val) in enumerate(zip(eps_labels, rescue_vals)):
    axes[0].text(i, val + 1.5, str(val), ha='center', va='bottom',
                 fontsize=11, fontweight='bold')
axes[0].set_title('Representative Rescue Count vs ε', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epsilon (ε)')
axes[0].set_ylabel('Rescued tweets (NMR → visible)')
axes[0].set_ylim(min(rescue_vals) - 20, max(rescue_vals) + 25)
axes[0].grid(color='#d1d5db', alpha=0.6)

# Right: distribution of bridge score shift ε=0.1 minus ε=1e-6
delta = sc_high['bridge_eps'] - sc_low['bridge_eps']
axes[1].hist(delta, bins=60, color='#7c3aed', alpha=0.75, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1.0, linestyle='--')
axes[1].set_title('Bridge Score Shift: ε=0.1 minus ε=1e-6\n(per note)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Δ bridge score')
axes[1].set_ylabel('Number of notes')
axes[1].grid(color='#d1d5db', alpha=0.6)

plt.tight_layout()
plt.show()